# 🛡️ Notebook 3: Bloom Filters in Front of a Database

The killer application of bloom filters is **protecting a slow backend from wasted lookups**.

## The scenario

You run a website. Users request profiles by username. Most requests hit valid users — but a **lot** of traffic is junk (typos, bots, scrapers scanning for usernames that don't exist). Every one of those invalid requests still travels all the way to your database, eats a connection, runs an index lookup, returns "not found", and occupies compute you could be using for real users.

**What if you could reject most nonexistent usernames in ~100 nanoseconds, before even touching the DB?**

That's a bloom filter.

## Learning objectives
- Use a bloom filter as a **negative cache** in front of a slow backend.
- Measure the latency and throughput improvement on a simulated workload.
- Understand the failure modes (false positives are okay, false negatives would be a bug).

## 🛠️ Setup

Pure Python — no Docker. Run `uv sync` and select the `.venv` kernel (`Cmd+Shift+P` → **Reload Window** if it doesn't appear).

## 🐢 A simulated slow "database"

Real DBs take ~1–10 ms per lookup over the network. We'll simulate that with `time.sleep`. The DB knows about 10,000 real usernames.

In [ ]:
import time
import random

class SlowDatabase:
    """A pretend database with a fixed 2 ms latency per lookup."""

    def __init__(self, known_usernames, latency_ms=2.0):
        self._users = set(known_usernames)
        self._latency = latency_ms / 1000.0
        self.lookup_count = 0

    def user_exists(self, username: str) -> bool:
        self.lookup_count += 1
        time.sleep(self._latency)  # simulate network + disk
        return username in self._users

random.seed(0)
real_users = {f"user_{i:05d}" for i in range(10_000)}
db = SlowDatabase(real_users, latency_ms=2.0)

print(f"DB has {len(real_users):,} real users, each lookup costs 2 ms")

## 🎯 The workload

We'll simulate 1,000 incoming requests where **only 10% are real users** and 90% are junk (typos, scans). This is realistic for public-facing APIs.

In [ ]:
def build_workload(real_users, total=1000, real_ratio=0.1):
    real_list = list(real_users)
    reqs = []
    for i in range(total):
        if random.random() < real_ratio:
            reqs.append(random.choice(real_list))           # hit
        else:
            reqs.append(f"ghost_{i}_{random.randint(0, 10**9)}")  # miss
    random.shuffle(reqs)
    return reqs

requests = build_workload(real_users, total=1000, real_ratio=0.1)
print(f"Generated {len(requests)} requests (≈10% real, 90% junk)")

## 🟥 Baseline: every request hits the DB

In [ ]:
db.lookup_count = 0
start = time.perf_counter()
for username in requests:
    db.user_exists(username)
baseline_seconds = time.perf_counter() - start

print(f"Total time:       {baseline_seconds*1000:.1f} ms")
print(f"DB lookups:       {db.lookup_count}")
print(f"Avg per request:  {baseline_seconds*1000/len(requests):.2f} ms")

Every request waits 2 ms for the DB, even the 90% that were never going to find anything. That's wasted time.

## 🟩 With a bloom filter as a shield

We pre-load a bloom filter with **every real username**. On each request we do:

1. Is this username in the bloom filter?
2. If **no** → it's definitely not a real user. Reject immediately, skip the DB. ⚡️
3. If **yes** → it's *probably* a real user. Ask the DB to be sure.

This is safe because bloom filters have **no false negatives** — if a real user exists, the filter will say yes. The only cost is false positives, where we pay an unnecessary DB lookup (same as the baseline).

In [ ]:
import math, hashlib
from pydantic import BaseModel, Field

class BloomParams(BaseModel):
    n: int = Field(gt=0)
    p: float = Field(gt=0, lt=1)
    @property
    def m(self): return int(math.ceil(-self.n * math.log(self.p) / (math.log(2) ** 2)))
    @property
    def k(self): return max(1, int(round((self.m / self.n) * math.log(2))))

class BloomFilter:
    def __init__(self, m, k):
        self.m, self.k = m, k
        self.bits = bytearray((m + 7) // 8)
    def _set_bit(self, i): self.bits[i // 8] |= 1 << (i % 8)
    def _get_bit(self, i): return (self.bits[i // 8] >> (i % 8)) & 1
    def _pos(self, x):
        d = hashlib.sha256(x.encode()).digest()
        h1 = int.from_bytes(d[:8], 'big'); h2 = int.from_bytes(d[8:16], 'big')
        for i in range(self.k):
            yield (h1 + i * h2) % self.m
    def add(self, x):
        for p in self._pos(x): self._set_bit(p)
    def __contains__(self, x):
        return all(self._get_bit(p) for p in self._pos(x))

# Size the filter for our 10,000 real users at 1% FPR
params = BloomParams(n=len(real_users), p=0.01)
shield = BloomFilter(m=params.m, k=params.k)
for u in real_users:
    shield.add(u)

print(f"Bloom shield: {params.m:,} bits ({params.m/8:.0f} bytes), k={params.k}")
print(f"Target FPR: 1% — at most ~1% of junk requests will still hit the DB")

In [ ]:
db.lookup_count = 0
rejected_by_shield = 0
start = time.perf_counter()
for username in requests:
    if username not in shield:
        rejected_by_shield += 1
        continue
    db.user_exists(username)
shielded_seconds = time.perf_counter() - start

print(f"Total time:          {shielded_seconds*1000:.1f} ms")
print(f"Rejected by filter:  {rejected_by_shield}")
print(f"DB lookups:          {db.lookup_count}")
print(f"Avg per request:     {shielded_seconds*1000/len(requests):.2f} ms")
print(f"\nSpeedup: {baseline_seconds / shielded_seconds:.1f}×")

## 📊 Side-by-side

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

ax1.bar(['no shield', 'bloom shield'], [baseline_seconds*1000, shielded_seconds*1000], color=['#c0392b', '#27ae60'])
ax1.set_ylabel('total time (ms)')
ax1.set_title('Wall-clock time for 1,000 requests')

ax2.bar(['no shield', 'bloom shield'], [1000, db.lookup_count], color=['#c0392b', '#27ae60'])
ax2.set_ylabel('DB lookups')
ax2.set_title('DB lookups performed')

plt.tight_layout()
plt.show()

## ✅ Correctness check: no request was wrongly rejected

This is the scary part. A false negative here would be a real bug: a legitimate user rejected before hitting the DB. But bloom filters **guarantee** no false negatives. Let's verify.

In [ ]:
real_requests = [r for r in requests if r in real_users]
wrongly_rejected = [r for r in real_requests if r not in shield]

# A false negative here = a real user turned away without the DB ever being asked.
# The data structure forbids it, so assert rather than eyeball a printed number.
assert wrongly_rejected == [], f"legit users wrongly rejected: {wrongly_rejected[:3]}"
print(f"Real users in workload:   {len(real_requests)}")
print(f"Wrongly rejected:         0  (guaranteed — bloom filters have no false negatives)")

# And the shield must actually have done something: every DB lookup we still paid
# for was either a real user or one of the ~1% false positives.
junk_requests = len(requests) - len(real_requests)
false_positive_lookups = db.lookup_count - len(real_requests)
print(f"Junk requests:            {junk_requests}")
print(f"  rejected by the filter: {rejected_by_shield}")
print(f"  leaked through (FP):    {false_positive_lookups} "
      f"({false_positive_lookups/junk_requests:.1%} of junk — budget was 1%)")
assert rejected_by_shield + false_positive_lookups == junk_requests
assert false_positive_lookups / junk_requests < 0.03, "FPR far above the 1% design target"

## 🧠 Where this pattern shows up in the real world

- **Cassandra, HBase, LevelDB, RocksDB** — every SSTable on disk has a bloom filter. Before reading a file from disk for a key, the filter is consulted; if it says "not here", the whole file is skipped.
- **CDN cache layers** — check a bloom filter before looking up an object in a distant origin.
- **Web crawlers** — "have I already visited this URL?" (our notebook 1 example, for real).
- **Chrome's Safe Browsing** — a bloom filter of known-malicious URLs ships with the browser so most checks are local.
- **Spell checkers of yore** — classic use case: "is this probably a real word?"

## 🎓 Takeaways

1. Bloom filters are a **fast rejector**. They are not a data store.
2. They shine when the cost of a backend miss is **much higher** than the cost of a bloom filter check, and most requests would have been misses anyway.
3. False positives waste a lookup (same cost as no bloom filter at all). False negatives would be a correctness bug — and luckily the data structure mathematically forbids them.
4. For long-lived systems, remember to **rebuild or resize** the filter if the real dataset grows well beyond the planned `n`.